# Multimodal Fashion & Context Retrieval - Demo

Thin demo over the `indexer/` and `retriever/` modules. Runs on a free Colab **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

Pipeline: FashionCLIP global + garment-region embeddings -> Chroma -> query decomposition -> compositional AND-scoring -> optional VQA re-rank.

This notebook reproduces every figure and number in the report.

## 1. Setup

In [ ]:
# In Colab, clone the repo first (skip if already cloned):
# !git clone https://github.com/manasa-manoj-nbr/fashion-retrieval.git
# %cd fashion-retrieval
!pip -q install -r requirements.txt

## 2. Fetch data + build the index
Downloads the official Fashionpedia val/test split (236 MB) and indexes 800 images. Data lives only in Colab; it is never committed to git.

In [ ]:
!python -m eval.fetch_data --n 800
!python -m eval.smoke_test          # ~30s sanity check before the full build
!python -m indexer.build --data data/images

## 3. Visual helpers

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from retriever.search import FashionRetriever
from retriever.rerank import VQAReranker
from retriever.query_parser import parse

retriever = FashionRetriever()
reranker = VQAReranker()   # local BLIP-VQA, no API key

def grid(query, k=5, only_global=False, rerank=False):
    res = retriever.search(query, k=k*4 if rerank else k, only_global=only_global)
    if rerank:
        res = reranker.rerank(parse(query), res)[:k]
    else:
        res = res[:k]
    fig, ax = plt.subplots(1, k, figsize=(3.2*k, 3.4))
    for a, r in zip(ax, res):
        a.imshow(Image.open(r.path)); a.axis('off'); a.set_title(f"{r.score:.2f}", fontsize=10)
    fig.suptitle(query, fontsize=13); plt.tight_layout(); plt.show()

def compare(query, k=5):
    """Global (vanilla-CLIP style) vs full pipeline, stacked - shows the binding lift."""
    g = retriever.search(query, k=k, only_global=True)
    f = retriever.search(query, k=k, only_global=False)
    fig, ax = plt.subplots(2, k, figsize=(3.2*k, 6.6))
    for row, (label, res) in enumerate([("GLOBAL (vanilla-CLIP)", g), ("FULL (ours)", f)]):
        for a, r in zip(ax[row], res):
            a.imshow(Image.open(r.path)); a.axis('off'); a.set_title(f"{r.score:.2f}", fontsize=9)
        ax[row][0].axis('on'); ax[row][0].set_xticks([]); ax[row][0].set_yticks([])
        ax[row][0].set_ylabel(label, fontsize=11)
    fig.suptitle(f'"{query}"  -  global vs full', fontsize=13); plt.tight_layout(); plt.show()

## 4. The 5 evaluation queries

In [ ]:
official = [
    'A person in a bright yellow raincoat.',
    'Professional business attire inside a modern office.',
    'Someone wearing a blue shirt sitting on a park bench.',
    'Casual weekend outfit for a city walk.',
    'A red tie and a white shirt in a formal setting.',
]
for q in official:
    grid(q, k=5)

## 5. Strengths gallery (colour / garment / zero-shot)

In [ ]:
for q in ['a navy blue blazer', 'a leather biker jacket', 'an elegant evening gown',
          'a floral summer dress', 'a pink hoodie', 'a monochrome black outfit']:
    grid(q, k=5)

## 6. Compositional binding: swap pairs
Same palette, opposite colour->garment binding. Note how the rank-1 image changes between the two, and the global-vs-full reordering.

In [ ]:
grid('a red top and blue pants', k=5)
grid('a blue top and red pants', k=5)
compare('a red top and blue pants')
compare('a white shirt and black pants')

## 7. Full query battery + quantitative evaluation

In [ ]:
!python -m eval.showcase
!python -m eval.evaluate